# SportRetail LAM — Extracción y Análisis de Datos
### Taller de Integración de APIs con Python y Power BI

**Flujo de trabajo:**  
Extracción desde API --> Transformación / Limpieza --> Exportación CSV --> Visualización en Power BI



 **Antes de correr este notebook** asegúrate de haber levantado el servidor local en el terminal de Visual o cmd:
- pip install fastapi uvicorn faker
- python api_server.py
El servidor corre en: http://localhost:8000/health

## 1. Importar librerías

In [157]:
import requests
import pandas as pd
import json

# Dirección base de la API local
BASE_URL = "http://localhost:8000"

print("Librerías cargadas correctamente")


Librerías cargadas correctamente


## 2. Verificar que la API esté activa

Antes de pedir datos, comprobamos que el servidor esté funcionando con el endpoint `/health`.


In [158]:
respuesta = requests.get(f"{BASE_URL}/health")

if respuesta.status_code == 200:
    print("API activa")
    print(json.dumps(respuesta.json(), indent=2))
else:
    print("Error al conectar. ¿Está corriendo en la terminal api_server.py?")


API activa
{
  "status": "ok",
  "records": {
    "products": 120,
    "users": 100,
    "carts": 200
  }
}


## 3. Extracción de datos — Productos

Consultamos **todos los productos** del catálogo usando el endpoint `/products`.  
Usamos el parámetro `limit=200` para traer todos los registros de una sola llamada.


In [159]:
respuesta_productos = requests.get(f"{BASE_URL}/products", params={"limit": 200})
data_productos = respuesta_productos.json()

print(f"Productos recibidos en esta llamada: {data_productos['count']}")


Productos recibidos en esta llamada: 120


In [160]:
# Convertir a DataFrame
df_productos = pd.DataFrame(data_productos["products"])

print("Datos: ")
df_productos.head(10)

Datos: 


,id,title,category,brand,price,discountPercentage,stock,rating,reviews,sku
0,1,PowerEdge Running Shoes v1,Footwear,PowerEdge,80.05,18.5,125.0,3.1,764,SR-FOO-0001
1,2,UrbanSport Running Shoes v2,Footwear,UrbanSport,74.56,10.5,15.0,NaN,248,SR-FOO-0002
2,3,AthletX Running Shoes v3,Footwear,AthletX,216.10,17.5,214.0,3.1,613,SR-FOO-0003
3,4,AthletX Running Shoes v4,Footwear,AthletX,212.08,8.5,79.0,3.0,791,SR-FOO-0004
4,5,SportRetail Pro Running Shoes v5,Footwear,SportRetail Pro,135.77,8.6,135.0,4.5,757,SR-FOO-0005
5,6,SpeedMax Basketball Shoes v1,Footwear,SpeedMax,72.73,7.3,321.0,4.0,380,SR-FOO-0006
6,7,SportRetail Pro Basketball Shoes v2,Footwear,SportRetail Pro,203.78,19.3,40.0,4.6,113,SR-FOO-0007
7,8,CoreFit Basketball Shoes v3,Footwear,CoreFit,91.60,8.9,343.0,3.2,709,SR-FOO-0008
8,9,AthletX Basketball Shoes v4,Footwear,AthletX,175.18,6.1,236.0,3.4,665,SR-FOO-0009
9,10,CoreFit Basketball Shoes v5,Footwear,CoreFit,244.64,19.4,117.0,4.6,333,SR-FOO-0010


## 4. Extracción de datos — Usuarios (Retailers)

Consultamos los retailers con el endpoint `/users`.


In [161]:
respuesta_usuarios = requests.get(f"{BASE_URL}/users", params={"limit": 200})
data_usuarios = respuesta_usuarios.json()

print(f"Usuarios recibidos en esta llamada: {data_usuarios['count']}")


Usuarios recibidos en esta llamada: 100


In [162]:

df_usuarios = pd.DataFrame(data_usuarios["users"])
df_usuarios.head(3)

,id,firstName,lastName,email,age,country,city,phone,address,retailerType
0,1,Valentino,Gomez,gonzalezvictoria@example.org,44,Argentina,Tucumán,+54 15 2496 0013,Calle Corrientes N° 784,Sports Chain
1,2,Rafael,Pérez,zfernandez@example.org,38,Colombia,Cartagena,573254235116,Diagonal 5ª # 8X-9,Sports Chain
2,3,Ángel,Osses,sperez@example.net,24,Chile,Santiago,+56 9 5210 3413,Calle Las Malvas 455 Of. 341,Online Retailer


## 5. Extracción de datos — Órdenes (Carts)

Consultamos las órdenes con el endpoint `/carts`.  


In [163]:
respuesta_ordenes = requests.get(f"{BASE_URL}/carts", params={"limit": 200})
data_ordenes = respuesta_ordenes.json()

print(f"Órdenes recibidas en esta llamada: {data_ordenes['count']}")


Órdenes recibidas en esta llamada: 200


In [164]:

df_ordenes = pd.DataFrame(data_ordenes["carts"])
df_ordenes.head(3)

,id,userId,totalProducts,products,orderDate,status,channel
0,1,65,2,"[{'id': 8, 'title': 'CoreFit Basketball Shoes ...",2024-12-03,delivered,Direct Sales
1,2,6,2,"[{'id': 91, 'title': 'CoreFit Resistance Bands...",2024-06-10,shipped,Distributor
2,3,87,5,"[{'id': 7, 'title': 'SportRetail Pro Basketbal...",2024-06-03,delivered,Direct Sales


In [165]:
filas = []
for orden in data_ordenes["carts"]:
    for producto in orden["products"]:
        filas.append({
            "orden_id":       orden["id"],
            "usuario_id":     orden["userId"],
            "fecha":          orden["orderDate"],
            "estado":         orden["status"],
            "canal":          orden["channel"],
            "producto_id":    producto["id"],
            "producto_nombre":producto["title"],
            "categoria":      producto["category"],
            "precio_unitario":producto["price"],
            "cantidad":       producto["quantity"],
            "descuento_pct":  producto["discountPercentage"],
        })

df_ordenes = pd.DataFrame(filas)
print(f"Filas generadas (líneas de orden): {len(df_ordenes)}")
df_ordenes.head(3)


Filas generadas (líneas de orden): 733


,orden_id,usuario_id,fecha,estado,canal,producto_id,producto_nombre,categoria,precio_unitario,cantidad,descuento_pct
0,1,65,2024-12-03,delivered,Direct Sales,8,CoreFit Basketball Shoes v3,Footwear,91.60,18.0,8.9
1,1,65,2024-12-03,delivered,Direct Sales,58,AthletX Sports Bra v3,Apparel,87.71,112.0,7.5
2,2,6,2024-06-10,shipped,Distributor,91,CoreFit Resistance Bands v1,Equipment,36.90,61.0,18.0


## 6. Exploración inicial — ¿Qué tenemos?

Miramos el tamaño, columnas y tipos de datos de cada tabla.


In [166]:
print("=== PRODUCTOS ===")
print(df_productos.shape)
print(df_productos.dtypes)


=== PRODUCTOS ===
(120, 10)
id                      int64
title                  object
category               object
brand                  object
price                 float64
discountPercentage    float64
stock                 float64
rating                float64
reviews                 int64
sku                    object
dtype: object


In [167]:
print("=== USUARIOS ===")
print(df_usuarios.shape)
print(df_usuarios.dtypes)
print()


=== USUARIOS ===
(100, 10)
id               int64
firstName       object
lastName        object
email           object
age              int64
country         object
city            object
phone           object
address         object
retailerType    object
dtype: object



In [169]:
print("=== ÓRDENES ===")
print(df_ordenes.shape)
print(df_ordenes.dtypes)

=== ÓRDENES ===
(733, 11)
orden_id             int64
usuario_id           int64
fecha               object
estado              object
canal               object
producto_id          int64
producto_nombre     object
categoria           object
precio_unitario    float64
cantidad           float64
descuento_pct      float64
dtype: object


## 7. Detección de nulos y duplicados

La API tiene  (valores nulos, duplicados).  
Aquí los identificamos antes de limpiarlos.


In [170]:
print("=== Productos ===")
print(df_productos.isnull().sum())

=== Productos ===
id                    0
title                 0
category              0
brand                 0
price                 0
discountPercentage    0
stock                 2
rating                4
reviews               0
sku                   0
dtype: int64


In [171]:
print("=== Usuarios ===")
print(df_usuarios.isnull().sum())

=== Usuarios ===
id              0
firstName       0
lastName        0
email           3
age             0
country         0
city            0
phone           0
address         0
retailerType    0
dtype: int64


In [172]:
print("=== Órdenes ===")
print(df_ordenes.isnull().sum())

=== Órdenes ===
orden_id            0
usuario_id          0
fecha               0
estado              0
canal               0
producto_id         0
producto_nombre     0
categoria           0
precio_unitario     0
cantidad           12
descuento_pct       0
dtype: int64


In [173]:
df_productos.duplicated(subset=['id']).sum()

np.int64(0)

In [174]:
df_usuarios.duplicated(subset=['id']).sum()

np.int64(0)

In [175]:
df_ordenes.duplicated(subset=['orden_id','producto_id']).sum()

np.int64(0)

## 8. Limpieza y transformación

Resolvemos cada problema detectado y lo documentamos.

In [176]:
# PRODUCTOS
# 1. Rellenar stock y rating nulo con 0
df_products_tra = df_productos["stock"] = df_productos["stock"].fillna(0)
df_products_tra = df_productos["rating"] = df_productos["rating"].fillna(0)
print(f"Stock nulo → rellenado con 0 unidades")

Stock nulo → rellenado con 0 unidades


In [177]:
print("=== Productos ===")
print(df_productos.isnull().sum())

=== Productos ===
id                    0
title                 0
category              0
brand                 0
price                 0
discountPercentage    0
stock                 0
rating                0
reviews               0
sku                   0
dtype: int64


In [178]:
before = len(df_productos)
df_productos = df_productos.drop_duplicates(subset=["id"], keep="first")
print(f"Duplicados eliminados: {before - len(df_productos)} filas")

Duplicados eliminados: 0 filas


In [179]:
#  ÓRDENES
# 1. Rellenar cantidad  nulo con 0
df_ordenes["cantidad"] = df_ordenes["cantidad"].fillna(0)
print(f"Cantidad nula → rellenada con 0 unidades")

Cantidad nula → rellenada con 0 unidades


In [180]:
print("=== Órdenes ===")
print(df_ordenes.isnull().sum())

=== Órdenes ===
orden_id           0
usuario_id         0
fecha              0
estado             0
canal              0
producto_id        0
producto_nombre    0
categoria          0
precio_unitario    0
cantidad           0
descuento_pct      0
dtype: int64


In [181]:
# 6. Convertir fecha a tipo datetime
df_ordenes["fecha"] = pd.to_datetime(df_ordenes["fecha"])
print("Columna 'fecha' convertida a datetime")


Columna 'fecha' convertida a datetime


## 9. Crear columnas calculadas

Añadimos métricas útiles para el análisis en Power BI.


In [182]:
# Precio final con descuento aplicado
df_ordenes["precio_final"] = (
    df_ordenes["precio_unitario"] * (1 - df_ordenes["descuento_pct"] / 100)
).round(2)

# Subtotal por línea de orden
df_ordenes["subtotal"] = (df_ordenes["precio_final"] * df_ordenes["cantidad"]).round(2)

# Año y mes para análisis temporal
df_ordenes["anio"] = df_ordenes["fecha"].dt.year
df_ordenes["mes"]  = df_ordenes["fecha"].dt.month
df_ordenes["mes_nombre"] = df_ordenes["fecha"].dt.strftime("%B")

print("Columnas nuevas agregadas a df_ordenes:")
print(["precio_final", "subtotal", "anio", "mes", "mes_nombre"])
df_ordenes[["orden_id","producto_nombre","precio_unitario","descuento_pct","precio_final","cantidad","subtotal"]].head(4)


Columnas nuevas agregadas a df_ordenes:
['precio_final', 'subtotal', 'anio', 'mes', 'mes_nombre']


,orden_id,producto_nombre,precio_unitario,descuento_pct,precio_final,cantidad,subtotal
0,1,CoreFit Basketball Shoes v3,91.60,8.9,83.45,18.0,1502.10
1,1,AthletX Sports Bra v3,87.71,7.5,81.13,112.0,9086.56
2,2,CoreFit Resistance Bands v1,36.90,18.0,30.26,61.0,1845.86
3,2,CoreFit Sports Bra v2,64.26,7.4,59.50,83.0,4938.50


## 10. Análisis exploratorio rápido

Algunos resúmenes antes de exportar para entender los datos.


In [183]:
print("=== TOP 5 Categorías por ventas totales ===")
ventas_cat = (df_ordenes.groupby("categoria")["subtotal"]
              .sum().sort_values(ascending=False).head(5))
print(ventas_cat.to_string())

print("=== Ventas por estado de orden ===")
print(df_ordenes.groupby("estado")["subtotal"].sum().sort_values(ascending=False).to_string())

print("=== Ventas por canal ===")
print(df_ordenes.groupby("canal")["subtotal"].sum().sort_values(ascending=False).to_string())


=== TOP 5 Categorías por ventas totales ===
categoria
Footwear       1774642.97
Apparel         701535.35
Equipment       647518.89
Accessories     577689.07
=== Ventas por estado de orden ===
estado
delivered    1629091.08
shipped       878633.34
confirmed     718227.49
cancelled     475434.37
=== Ventas por canal ===
canal
E-commerce B2B    1389395.66
Direct Sales      1166877.22
Distributor       1145113.40


In [184]:
print("=== Resumen de países de retailers ===")
print(df_usuarios["country"].value_counts().to_string())

print("=== Tipo de retailer ===")
print(df_usuarios["retailerType"].value_counts().to_string())


=== Resumen de países de retailers ===
country
Colombia     21
Perú         21
México       20
Argentina    19
Chile        19
=== Tipo de retailer ===
retailerType
Sports Chain        24
Independent         24
Superstore          20
Online Retailer     19
Department Store    13


## 11. Dataset enriquecido — Unir órdenes con información de usuarios

Hacemos un JOIN para añadir país y tipo de retailer a cada línea de orden.  
Esto enriquece el dataset para los filtros de Power BI.


In [185]:
# Seleccionamos solo las columnas relevantes de usuarios
usuarios_slim = df_usuarios[["id","country","city","retailerType"]].rename(
    columns={"id": "usuario_id"}
)

# Unir con órdenes
df_final = df_ordenes.merge(usuarios_slim, on="usuario_id", how="left")

print(f"Filas en dataset final: {len(df_final)}")
print(f"Columnas: {list(df_final.columns)}")
df_final.head(3)


Filas en dataset final: 733
Columnas: ['orden_id', 'usuario_id', 'fecha', 'estado', 'canal', 'producto_id', 'producto_nombre', 'categoria', 'precio_unitario', 'cantidad', 'descuento_pct', 'precio_final', 'subtotal', 'anio', 'mes', 'mes_nombre', 'country', 'city', 'retailerType']


,orden_id,usuario_id,fecha,estado,canal,producto_id,producto_nombre,categoria,precio_unitario,cantidad,descuento_pct,precio_final,subtotal,anio,mes,mes_nombre,country,city,retailerType
0,1,65,2024-12-03,delivered,Direct Sales,8,CoreFit Basketball Shoes v3,Footwear,91.60,18.0,8.9,83.45,1502.10,2024,12,December,Argentina,Tucumán,Department Store
1,1,65,2024-12-03,delivered,Direct Sales,58,AthletX Sports Bra v3,Apparel,87.71,112.0,7.5,81.13,9086.56,2024,12,December,Argentina,Tucumán,Department Store
2,2,6,2024-06-10,shipped,Distributor,91,CoreFit Resistance Bands v1,Equipment,36.90,61.0,18.0,30.26,1845.86,2024,6,June,Chile,Valparaíso,Independent


## 12. Exportar CSV para Power BI



In [187]:
import os

# Crear carpeta de salida si no existe
os.makedirs("output", exist_ok=True)

# Tablas individuales
df_productos.to_csv("output/productos.csv", index=False, encoding="utf-8-sig")
df_usuarios.to_csv("output/usuarios.csv",   index=False, encoding="utf-8-sig")
df_ordenes.to_csv("output/ordenes.csv",     index=False, encoding="utf-8-sig")

print("Archivos exportados en la carpeta /output:")
for f in os.listdir("output"):
    size = os.path.getsize(f"output/{f}")
    print(f"{f} ({size/1024:.1f} KB)")


Archivos exportados en la carpeta /output:
carts.json (199.1 KB)
ordenes.csv (89.7 KB)
productos.csv (10.3 KB)
products.json (34.1 KB)
users.json (32.3 KB)
usuarios.csv (11.7 KB)
ventas_consolidado.csv (111.1 KB)
